In [1]:
import sys
import os
sys.path.append('..')

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json

import utilities.functions as functions
from utilities.functions import (
    retidos,
    calcula_viabilidade,
    analisar_retencao,
)

from utilities.descritiva import  (
    matriz_migracao,
    cria_base_decil_wide
)

In [2]:
import os
import os
os.getcwd()
import os
import pandas as pd
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 


In [3]:

publico_janeiro_dezembro = pd.read_parquet(BASE_PATH / "gold" / "publico_janeiro_dezembro.parquet")


In [4]:
publico_janeiro_dezembro['num_pedidos_hist'].unique()

array([ 6,  4,  8,  5,  3,  9,  7, 10])

In [5]:
import json
import os

# Se você quer apenas a lista de customer_ids
meu_dict = {'amostra_aleatoria': amostra_aleatoria['customer_id'].tolist()}

# Criar pasta se não existir
os.makedirs('dados/gold', exist_ok=True)

# Salvar JSON
with open('dados/gold/amostra_aleatoria.json', 'w') as f:
    json.dump(meu_dict, f, indent=4)
    
print("JSON salvo com sucesso!")

NameError: name 'amostra_aleatoria' is not defined

In [ ]:
dois_pedidos=publico_janeiro_dezembro[publico_janeiro_dezembro['num_pedidos_hist']==3]
dois_pedidos

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio,pedidos_sum,outlier,decil
10,0191736095785dbb97a07cd153a5b006b6ab5fc438389d...,control,12,1,3,87.90,87.900,1,False,7
11,0191736095785dbb97a07cd153a5b006b6ab5fc438389d...,control,1,2,3,153.60,76.800,2,False,10
16,01fee5ac2b93fed4a7b35cb1618793c2ad2c25509a47bf...,target,12,1,3,39.99,39.990,1,False,2
17,01fee5ac2b93fed4a7b35cb1618793c2ad2c25509a47bf...,target,1,2,3,95.79,47.895,2,False,8
18,02849fcc82a7201718d42d117e4db5f7fcc801a4807cbe...,control,12,1,3,38.50,38.500,1,False,2
...,...,...,...,...,...,...,...,...,...,...
403,fcd1a726c8f1e23d0e29e95325b62bb23c00c4ab1a838f...,target,1,2,3,68.85,34.425,2,False,6
404,fe06dde1320fd0fec1857b8a375a0a56bef4c0da4e73dd...,control,12,1,3,43.00,43.000,1,False,3
405,fe06dde1320fd0fec1857b8a375a0a56bef4c0da4e73dd...,control,1,2,3,93.00,46.500,2,False,8
414,ff4770f539de10af59e9f33b00fa740820b4b770c28692...,target,12,1,3,19.00,19.000,1,False,1


In [ ]:
dois_pedidos=publico_janeiro_dezembro[publico_janeiro_dezembro['pedidos_sum']=='2']['customer_id'].unique()
amostra_aleatoria =publico_janeiro_dezembro[publico_janeiro_dezembro['customer_id'].isin(dois_pedidos)]
df_stats_mes = amostra_aleatoria.groupby(['order_created_month', 'pedidos_sum']).agg(
    total_clientes=('customer_id', 'nunique')
).round(2)
matriz_migracao(amostra_aleatoria,mes_0=12,mes_1=1,group_by_extra='pedidos_sum')

,mes_12,mes_1,is_target,pedidos_sum,total_clientes
0,0,1,control,2,23
1,0,1,control,3,8
2,0,1,control,4,10
3,0,1,control,5,4
4,0,1,control,6,2
5,0,1,control,7,2
6,0,1,target,2,45
7,0,1,target,3,14
8,0,1,target,4,10
9,0,1,target,5,5


Calculando retencao considerando dois ou + pedidos

In [ ]:
df_1,clientes_retidos_1=retidos(publico_janeiro_dezembro, mes0=12, mes1=1,pedidos=1)
print(clientes_retidos_1)

In [ ]:
retencao_pedido_1=analisar_retencao(df_1)
retencao_pedido_1

Calculando retencao considerando tres ou + pedidos

In [ ]:
df_2,clientes_retidos_2=retidos(publico_janeiro_dezembro, mes0=12, mes1=1,pedidos=2)
print(clientes_retidos_2)

In [ ]:
retencao_pedido_2=analisar_retencao(df_2)
retencao_pedido_2

Viabilidade 

In [ ]:
resultados, agg = calcula_viabilidade(
    publico_janeiro_dezembro,
    mes_campanha=12,
    mes_seguinte=1,
    coupon_value=10.0,   
    margin_rate=0.12     
)

resultados


Calculando retencao separadamente para outliers

In [ ]:
orderns_de=publico_janeiro_dezembro[publico_janeiro_dezembro['order_created_month']==12]

In [ ]:
distance = 1.5 * (np.nanpercentile(orderns_de['total_amount_mes'], 75) - np.nanpercentile(orderns_de['total_amount_mes'], 25))
lim_sup=distance + np.nanpercentile(orderns_de['total_amount_mes'], 75)

In [ ]:
red_square = dict(markerfacecolor='r', markeredgecolor='r', marker='.')
orderns_de['total_amount_mes'].plot(kind='box', xlim=(0, 500), vert=False, flierprops=red_square, figsize=(16,2))

In [ ]:
orderns_de['total_amount_mes'].describe().round(2)

In [ ]:
pb_drop_dj = publico_janeiro_dezembro.drop(
    publico_janeiro_dezembro[
        (publico_janeiro_dezembro['total_amount_mes'] == 0) |
        (publico_janeiro_dezembro['total_amount_mes'] > lim_sup)
    ].index,
    axis=0
)

In [ ]:
pb_drop_dj

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=pb_drop_dj,
    x='order_created_month',               
    y='total_amount_mes',   
    hue='is_target',           
    showfliers=False          
)

plt.title('Distribuição do valor gasto por mês e target')
plt.xlabel('Mês')
plt.ylabel('Total Gasto no Mês')
plt.legend(title='Grupo')
plt.tight_layout()
plt.show()

In [ ]:
pb_drop_dj_1,clientes_retidos_1=retidos(pb_drop_dj, mes0=12, mes1=1,pedidos=1)
print(clientes_retidos_1)

In [ ]:
pb_drop_dj_s=analisar_retencao(pb_drop_dj_1)
pb_drop_dj_s

In [ ]:
resultados_out, agg_out = calcula_viabilidade(
    pb_drop_dj,
    mes_campanha=12,
    mes_seguinte=1,
    coupon_value=10.0,   
    margin_rate=0.12     
)

resultados_out
